In [ ]:
import pandas as pd

df = pd.read_csv("hindi_english_parallel.csv")
df.dropna(inplace=True)
df["english"] = df["english"].str.strip()
df["hindi"] = df["hindi"].str.strip()

df = df[
    (df["english"].str.len() < 200) &
    (df["hindi"].str.len() < 200)
]

df = df.sample(n=100000, random_state=42).reset_index(drop=True)

In [64]:
train_df = df.sample(frac=0.9, random_state=42)
val_df = df.drop(train_df.index)

print(len(train_df), len(val_df))

90000 10000


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    MarianTokenizer,
    MarianMTModel,
    MarianConfig
)
from tqdm import tqdm
import sacrebleu

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"
TEACHER_DIR = "teacher/checkpoint-5064"   

In [ ]:
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)

In [67]:
teacher = MarianMTModel.from_pretrained(TEACHER_DIR).to(DEVICE)
teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False

In [ ]:
teacher_config = teacher.config

student_config = MarianConfig(
    vocab_size=teacher_config.vocab_size,
    d_model=teacher_config.d_model,
    encoder_layers=3,          
    decoder_layers=3,
    encoder_attention_heads=teacher_config.encoder_attention_heads,
    decoder_attention_heads=teacher_config.decoder_attention_heads,
    encoder_ffn_dim=teacher_config.encoder_ffn_dim,
    decoder_ffn_dim=teacher_config.decoder_ffn_dim,
    max_position_embeddings=teacher_config.max_position_embeddings,
    pad_token_id=teacher_config.pad_token_id,
    bos_token_id=teacher_config.bos_token_id,
    eos_token_id=teacher_config.eos_token_id,
)

student = MarianMTModel(student_config).to(DEVICE)

In [69]:
student = MarianMTModel(student_config).to("cuda")

In [ ]:
with torch.no_grad():
    student.model.shared.load_state_dict(
        teacher.model.shared.state_dict()
    )
    enc_map = [0, 2, 5]
    for s_idx, t_idx in enumerate(enc_map):
        student.model.encoder.layers[s_idx].load_state_dict(
            teacher.model.encoder.layers[t_idx].state_dict()
        )
    dec_map = [0, 2, 5]
    for s_idx, t_idx in enumerate(dec_map):
        student.model.decoder.layers[s_idx].load_state_dict(
            teacher.model.decoder.layers[t_idx].state_dict()
        )

In [71]:
def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

print("Teacher params:", count_params(teacher), "M")
print("Student params:", count_params(student), "M")


Teacher params: 76.381184 M
Student params: 54.311936 M


In [72]:
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

In [ ]:
from torch.utils.data import Dataset
import torch

class TranslationDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.src = df["english"].tolist()
        self.tgt = df["hindi"].tolist()

        self.tokenizer = tokenizer
        self.max_len = max_len

        assert len(self.src) == len(self.tgt), "Source and target length mismatch"

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src_text = self.src[idx]
        tgt_text = self.tgt[idx]

        x = self.tokenizer(
            src_text,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        with self.tokenizer.as_target_tokenizer():
            y = self.tokenizer(
                tgt_text,
                max_length=self.max_len,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

        labels = y["input_ids"].squeeze(0)

        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": x["input_ids"].squeeze(0),
            "attention_mask": x["attention_mask"].squeeze(0),
            "labels": labels
        }

In [78]:
train_dataset = TranslationDataset(train_df, tokenizer)
val_dataset   = TranslationDataset(val_df, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [79]:
optimizer = torch.optim.AdamW(student.parameters(), lr=3e-5)

In [11]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [33]:
DEVICE = "cuda"

In [ ]:
def get_alpha(epoch):
    if epoch < 3:
        return 0.7      
    else:
        return 0.4      

In [ ]:
import torch.nn.functional as F
from tqdm import tqdm

T = 2.0

for epoch in range(6):
    student.train()
    total_loss = 0.0
    alpha = get_alpha(epoch)
    if epoch == 3:   
        for g in optimizer.param_groups:
            g["lr"] = 1e-5

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.no_grad():
            teacher_out = teacher(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            teacher_logits = teacher_out.logits

        student_out = student(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        student_logits = student_out.logits

        ce_loss = student_out.loss

        log_p_student = F.log_softmax(student_logits / T, dim=-1)
        p_teacher = F.softmax(teacher_logits / T, dim=-1)

        kd_mask = (labels != -100).unsqueeze(-1)  

        log_p_student = log_p_student * kd_mask
        p_teacher = p_teacher * kd_mask

        kd_loss = F.kl_div(
            log_p_student,
            p_teacher,
            reduction="sum"
        )

        kd_loss = kd_loss / kd_mask.sum()
        kd_loss = kd_loss * (T * T)

        loss = alpha * kd_loss + (1 - alpha) * ce_loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


In [96]:
def evaluate_bleu(model, dataloader):
    model.eval()
    preds, refs = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating BLEU"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)

            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=128,
                num_beams=4
            )

            decoded_preds = tokenizer.batch_decode(
                generated, skip_special_tokens=True
            )

            labels = batch["labels"]
            labels = torch.where(
                labels != -100,
                labels,
                tokenizer.pad_token_id
            )

            decoded_refs = tokenizer.batch_decode(
                labels, skip_special_tokens=True
            )

            preds.extend(decoded_preds)
            refs.extend([[r] for r in decoded_refs])

    return sacrebleu.corpus_bleu(preds, refs).score


In [97]:
bleu = evaluate_bleu(student, val_loader)
print("Student BLEU:", bleu)

/workspace/exp_armaan/env/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/workspace/exp_armaan/env/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(

valuating BLEU: 100%|██████████| 625/625 [02:14<00:00,  4.63it/s]

Student BLEU: 24.808415001701817


In [ ]:
student.save_pretrained("student")
tokenizer.save_pretrained("student")